# URLvestigia retrieval evaluation

Measures **availability, coverage, and overlap** on two axes:

* across the `backend` engines within `ddgs`, so the fallback chain in
  `retrieval/urlvestigia.py` is ordered on evidence rather than habit;
* across the four **providers**, which is a different question — they are separate
  corpora, so the interesting number is how little they overlap.

Run this before changing a retrieval default, and paste the summary tables into
`governance/model_cards/urlvestigia-retrieval.md`.

**This notebook hits live services.** It is intentionally small. Widening it will
get you rate-limited, which is itself one of the things being measured.

Set `URLVESTIGIA_CONTACT` before running: Wikipedia, OpenAlex, and arXiv give identified
callers better rate limits, and an unset contact will look like a flaky provider.

In [ ]:
import os, sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # retrieval/ - so `import urlvestigia` resolves
from urlvestigia import text_to_urls

os.environ.setdefault("URLVESTIGIA_CONTACT", "")  # set to a team address before running

ENGINES = ["duckduckgo", "yahoo", "startpage", "yandex"]
PROVIDERS = ["ddgs", "wikipedia", "openalex", "arxiv"]

# Probe queries: one navigational, one informational, one long-tail.
QUERIES = [
    "Cloudera CDP use cases",
    "iceberg table maintenance best practices",
    "how to throttle a metasearch client politely",
]

# The queries above are enterprise/web-flavoured and would score the scholarly
# providers near zero for reasons that say nothing about availability. Provider
# comparison needs subjects all four corpora actually cover.
PROVIDER_QUERIES = [
    "apache iceberg table format",
    "retrieval augmented generation",
    "differential privacy",
]

MAX_RESULTS = 10
COOLDOWN_S = 2  # be a good citizen between calls

## 1. Collect results per engine

Each engine is queried on its own so a throttled engine shows up as a zero rather
than being silently masked by the fallback chain.

In [ ]:
results = {}   # (query, engine) -> list[url]
errors = {}    # (query, engine) -> str

for query in QUERIES:
    for engine in ENGINES:
        try:
            urls = text_to_urls(query, max_results=MAX_RESULTS, backend=engine)
            results[(query, engine)] = urls
        except Exception as exc:
            results[(query, engine)] = []
            errors[(query, engine)] = f"{type(exc).__name__}: {exc}"
        time.sleep(COOLDOWN_S)

print(f"{len(results)} engine/query pairs, {len(errors)} errors")
for key, msg in errors.items():
    print("  ", key, "→", msg)

## 2. Availability and yield

**Availability** = fraction of queries that returned anything at all. This is the
number that should drive fallback order — an engine with great results that answers
half the time belongs behind a duller, reliable one.

**Yield** = mean URLs returned, against a ceiling of `MAX_RESULTS`.

In [ ]:
print(f"{'engine':<12} {'availability':>13} {'mean yield':>11}")
print("-" * 38)
for engine in ENGINES:
    counts = [len(results[(q, engine)]) for q in QUERIES]
    availability = sum(1 for c in counts if c) / len(counts)
    print(f"{engine:<12} {availability:>12.0%} {sum(counts) / len(counts):>11.1f}")

## 3. Overlap between engines

Jaccard similarity of returned URL sets, pooled across queries. **Low overlap is
the argument for a longer fallback chain** — it means adding an engine genuinely
widens coverage instead of re-returning what DuckDuckGo already found.

In [ ]:
def jaccard(a, b):
    if not a and not b:
        return float("nan")
    return len(a & b) / len(a | b)

pooled = {e: {u for q in QUERIES for u in results[(q, e)]} for e in ENGINES}

print(f"{'':<12}" + "".join(f"{e[:9]:>11}" for e in ENGINES))
for a in ENGINES:
    row = "".join(f"{jaccard(pooled[a], pooled[b]):>11.2f}" for b in ENGINES)
    print(f"{a:<12}{row}")

union = set().union(*pooled.values()) if pooled else set()
ddg = pooled.get("duckduckgo", set())
if union:
    print(f"\nunique URLs across all engines: {len(union)}")
    print(f"found by duckduckgo alone:      {len(ddg)} ({len(ddg) / len(union):.0%} of union)")

## 4. Does the fallback chain actually help?

Compares the default single engine against the full chain on the same queries.
If the chain does not raise the fill rate, the extra engines are cost without
benefit and the default should stay narrow.

In [ ]:
chain = ",".join(ENGINES)

for query in QUERIES:
    solo = len(results[(query, "duckduckgo")])
    try:
        chained = len(text_to_urls(query, max_results=MAX_RESULTS, backend=chain))
    except Exception as exc:
        chained = f"error ({type(exc).__name__})"
    print(f"{query[:46]:<48} duckduckgo={solo:<3} chain={chained}")
    time.sleep(COOLDOWN_S)

## 5. Provider availability

The measurement the provider seam exists for. `ddgs` reaches four engines by one
mechanism, so it is the row that can be blocked by IP reputation; the other three
are documented APIs that should answer from anywhere, including the datacenter
address a CDP deployment presents.

If that pattern ever inverts, the argument for the API providers has weakened and
the model card should say so.

In [ ]:
provider_results = {}   # (query, provider) -> list[url]
provider_errors = {}

for query in PROVIDER_QUERIES:
    for provider in PROVIDERS:
        try:
            provider_results[(query, provider)] = text_to_urls(
                query, provider=provider, max_results=MAX_RESULTS)
        except Exception as exc:
            provider_results[(query, provider)] = []
            provider_errors[(query, provider)] = f"{type(exc).__name__}: {exc}"
        time.sleep(COOLDOWN_S)

print(f"{'provider':<12} {'availability':>13} {'mean yield':>11}")
print("-" * 38)
for provider in PROVIDERS:
    counts = [len(provider_results[(q, provider)]) for q in PROVIDER_QUERIES]
    availability = sum(1 for c in counts if c) / len(counts)
    print(f"{provider:<12} {availability:>12.0%} {sum(counts) / len(counts):>11.1f}")

for key, msg in provider_errors.items():
    print("  error:", key, "-", msg)

## 6. Overlap between providers

Read this the opposite way to the engine overlap above.

For engines, **low overlap argues for a longer fallback chain** — another engine
widens coverage. For providers, low overlap is expected and is the argument for
*offering* them, while simultaneously being the reason they are **not** chained:
a corpus that returns nothing in common with the web is not a substitute for it
when the web is throttled. Falling through would answer a product question with
preprints and look like success.

Near-zero off-diagonal numbers here confirm the design in `docs/ARCHITECTURE.md`
under *Providers are an axis, not links in the chain*. Numbers close to 1.0 would
mean a provider is redundant and should be dropped.

In [ ]:
pooled_by_provider = {
    p: {u for q in PROVIDER_QUERIES for u in provider_results[(q, p)]}
    for p in PROVIDERS
}

print(f"{'':<12}" + "".join(f"{p[:9]:>11}" for p in PROVIDERS))
for a in PROVIDERS:
    row = "".join(f"{jaccard(pooled_by_provider[a], pooled_by_provider[b]):>11.2f}"
                  for b in PROVIDERS)
    print(f"{a:<12}{row}")

union = set().union(*pooled_by_provider.values()) if pooled_by_provider else set()
web = pooled_by_provider.get("ddgs", set())
if union:
    print(f"\nunique URLs across all providers: {len(union)}")
    print(f"found by ddgs alone:              {len(web)} ({len(web) / len(union):.0%} of union)")
    print("\nThe remainder is coverage no web engine returned at all.")

## 7. Do the per-provider options actually apply?

An option a provider accepts and ignores is worse than one it rejects: the result
set looks successful and the stored record claims a filter that never ran. arXiv
did exactly this before the term clause was parenthesised — `submittedDate` bound
to the last word of a multi-word query only, so "the past month" returned papers
from 2013.

These checks are cheap and worth repeating whenever a provider's API changes.

In [ ]:
# Wikipedia: region must select the language edition, not re-rank one corpus.
for region, host in [("wt-wt", "en"), ("kr-kr", "ko"), ("jp-jp", "ja")]:
    urls = text_to_urls("apache iceberg", provider="wikipedia",
                        max_results=3, region=region)
    ok = urls and all(u.startswith(f"https://{host}.wikipedia.org/") for u in urls)
    print(f"wikipedia region={region:<6} -> {host}.wikipedia.org  {'OK' if ok else 'CHECK'}")
    time.sleep(COOLDOWN_S)

# arXiv and OpenAlex: a recency window must actually narrow the result set.
for provider in ["arxiv", "openalex"]:
    wide = text_to_urls("apache iceberg", provider=provider, max_results=10)
    time.sleep(COOLDOWN_S)
    narrow = text_to_urls("apache iceberg", provider=provider,
                          max_results=10, timelimit="m")
    shared = len(set(wide) & set(narrow))
    print(f"{provider:<10} any-time={len(wide):<3} past-month={len(narrow):<3} "
          f"shared={shared:<3} {'OK' if set(narrow) != set(wide) else 'CHECK - filter may be ignored'}")
    time.sleep(COOLDOWN_S)

## 8. Record the finding

Copy the availability and overlap tables into
`governance/model_cards/urlvestigia-retrieval.md` under *Evaluation*, with today's date.
Behaviour drifts — an undated measurement is not evidence.

Record both axes. Engine availability drives the `backend` chain order; provider
availability is what tells you whether the API providers are still doing the job
they were added for, which is answering when the web engines will not.

If a default changes as a result, note it in the model card's *Change log* and
update the tables in `retrieval/README.md`.